# W3.1 — Cosine baseline validation

**Chủ sở hữu:** B. Phạm vi tuần 2–4, độc lập branch A.

**Input:** W2.2 split, W2.3 audit pass, cosine_baseline.json.

Cosine không học tham số. Score liên tục giữ thông tin xếp hạng cho ROC-AUC/AP.
Các phép thử toy chạy trước dữ liệu thật. Không dùng nhãn node làm feature.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p/'configs').is_dir() and (p/'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ repo hoặc thư mục con có configs/ và notebooks/.')

In [ ]:
get_ipython().run_line_magic('run', '"' + (PROJECT_ROOT/'notebooks/shared/runtime.ipynb').as_posix() + '"')
get_ipython().run_line_magic('run', '"' + (PROJECT_ROOT/'notebooks/shared/graph_ops.ipynb').as_posix() + '"')
get_ipython().run_line_magic('run', '"' + (PROJECT_ROOT/'notebooks/shared/metrics.ipynb').as_posix() + '"')

D:\gnn-node-link-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\gnn-node-link-prediction\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
config = read_json(PROJECT_ROOT/'configs/cosine_baseline.json')
print(config)

{'split_id': 'cora_lp_v1', 'seed': 42, 'feature_preprocessing': 'raw_float64', 'epsilon': 1e-12, 'chunk_size': 256, 'evaluation_split': 'validation', 'atol': 1e-12, 'rtol': 1e-10}


In [ ]:
toy_x = np.array([[1.,0.],[1.,0.],[0.,1.],[0.,0.]])
toy_pairs = np.array([[0,0,0,1],[1,2,3,0]])
np.testing.assert_allclose(cosine_scores(toy_x,toy_pairs),[1,0,0,1])
assert lp_metrics([0,1],[0,1])['roc_auc'] == 1
assert lp_metrics([0,1],[1,0])['roc_auc'] == 0
assert lp_metrics([0,1],[1,0])['average_precision'] == .5
assert lp_metrics([0,1],[1,1])['roc_auc'] == .5
assert lp_metrics([0,1],[1,1])['average_precision'] == .5
checks = {'toy_cosine_and_rank_metrics':{'status':'passed','observed':[1,0,0,1]}}
checks['empty_pairs'] = expect_error(lambda:cosine_scores(toy_x,np.empty((2,0),dtype=int)))
checks['out_of_range'] = expect_error(lambda:cosine_scores(toy_x,np.array([[0],[4]])))
checks['nan_features'] = expect_error(lambda:cosine_scores(np.array([[np.nan],[1]]),np.array([[0],[1]])))
checks['inf_scores'] = expect_error(lambda:lp_metrics([0,1],[0,np.inf]))
checks['empty_metric'] = expect_error(lambda:lp_metrics([],[]))
checks['label_score_mismatch'] = expect_error(lambda:lp_metrics([0,1],[1]))
checks['single_class'] = expect_error(lambda:lp_metrics([1,1],[0,1]))

In [ ]:
data_config = read_json(PROJECT_ROOT/'configs/data_protocol.json')
data = load_cora(PROJECT_ROOT,data_config)
full_pairs = canonicalize(data.edge_index.numpy(),data.num_nodes)
fingerprints = {'features':array_hash(data.x.numpy()),'full_pairs':array_hash(full_pairs)}

In [ ]:
split_dir = PROJECT_ROOT/'artifacts/splits'/data_config['split_id']
manifest = read_json(split_dir/'manifest.json')
if not (split_dir/'edges.npz').exists():
    raise FileNotFoundError('Thiếu edges.npz: chạy W2.2 trước.')
with np.load(split_dir/'edges.npz',allow_pickle=False) as archive:
    arrays = {k:archive[k] for k in archive.files}
assert manifest['input_fingerprints'] == fingerprints, 'Dataset fingerprint thay đổi'
assert manifest['config'] == data_config, 'Protocol thay đổi: dùng split ID mới'
assert mapping_hash(arrays) == manifest['split_hash'], 'Split hash sai'
assert {k:array_hash(v) for k,v in arrays.items()} == manifest['array_hashes']
assert manifest['counts'] == {k:v.shape[1] for k,v in arrays.items()}
assert manifest['num_nodes'] == data.num_nodes
train_adjacency = bidirectional(arrays['train_pos'],data.num_nodes)

In [ ]:
import time
audit = read_json(PROJECT_ROOT/'results/week02/leakage_audit.json')
assert audit['status'] == 'passed' and audit['split_hash'] == manifest['split_hash']
assert audit['input_fingerprints'] == fingerprints
audit_split(arrays,full_pairs,data.num_nodes,train_adjacency)
assert config['split_id'] == manifest['split_id'] and config['evaluation_split'] == 'validation'
assert config['feature_preprocessing'] == 'raw_float64'
pairs = np.concatenate([arrays['val_pos'],arrays['val_neg']],axis=1)
labels = np.concatenate([np.ones(arrays['val_pos'].shape[1]),np.zeros(arrays['val_neg'].shape[1])])
run = new_run(PROJECT_ROOT,'w3_cosine',config)
start = time.perf_counter()
scores = cosine_scores(data.x.numpy(),pairs,config['epsilon'],config['chunk_size'])
metrics = lp_metrics(labels,scores)
elapsed = time.perf_counter()-start
np.savez_compressed(run/'validation_predictions.npz',pairs=pairs,labels=labels,scores=scores)
previous_path = PROJECT_ROOT/'results/week03/cosine_validation.json'
if previous_path.exists():
    previous = read_json(previous_path)
    if previous['config'] == config and previous['split_hash'] == manifest['split_hash']:
        previous_file = PROJECT_ROOT/'runs'/previous['run_id']/'validation_predictions.npz'
        if previous_file.exists():
            with np.load(previous_file,allow_pickle=False) as old:
                np.testing.assert_allclose(scores,old['scores'],atol=config['atol'],rtol=config['rtol'])
                assert np.array_equal(labels,old['labels'])
            np.testing.assert_allclose([metrics[k] for k in ['roc_auc','average_precision']],
                                       [previous['validation'][k] for k in ['roc_auc','average_precision']],atol=config['atol'],rtol=config['rtol'])
            checks['independent_run_reproduction'] = {'status':'passed','previous_run_id':previous['run_id'],'atol':config['atol'],'rtol':config['rtol']}
result = publish_result(PROJECT_ROOT,run,'results/week03/cosine_validation.json',
    {'validation':metrics,'validation_metric':'roc_auc','validation_value':metrics['roc_auc'],'elapsed_seconds':elapsed,
     'input_fingerprints':fingerprints,'checks':checks,'evaluation_split':'validation'},config,3,'W3.1','cosine','Cora',manifest)
print(metrics); print('run_id:',run.name)

{'roc_auc': 0.8121701918489497, 'average_precision': 0.8248011097161196, 'n_positive': 263, 'n_negative': 263}
run_id: 20260928T070135_w3_cosine_a81fd148


## Kết luận và tự kiểm tra

Chỉ các assertion chạy thành công mới tạo kết quả bàn giao. Output bên trên chứa đường dẫn/run ID để truy vết. B chưa được đánh dấu đã tự review.

1. Cosine có tham số nào được học?
2. Tại sao không threshold score trước ROC-AUC?
3. AP khác diện tích PR hình thang như thế nào?
4. Tại sao chưa tính test metric?